# Swin-X2S — Inter-Comparison Baseline for PPCNet
## Adapted from: Liu et al., "Swin-X2S: Reconstructing 3D Shape from 2D Biplanar X-ray with Swin Transformers", arXiv 2025

### Adaptation Protocol (identical to 3D-ReVert / X2CT-GAN / BX2S-Net inter-comparison)
| Aspect | What we keep from Swin-X2S | What we keep from PPCNet v6 |
|---|---|---|
| **Encoder** | 2D Swin Transformer (Swin-T, ImageNet pretrained, 1-ch adapted) x 2 views | — |
| **Dim Expanding** | 2D features replicated along orthogonal axis + 1x1x1 3D conv per level | — |
| **Decoder** | 3D U-shaped decoder with cross-attention at bottleneck | — |
| **Loss** | DiceCE (single-view) + KL divergence (cross-view) | — |
| **Optimizer** | AdamW lr=3e-5, weight decay=5e-1, warmup cosine 20 epochs | — |
| **Dataset** | — | Lumbar_Filtered_1037 (829/103/105) |
| **Coordinate system** | — | pts_to_local / local_to_world / compute_scale |
| **GT representation** | Volume (occupancy at 64 cubed) from GT point cloud | GT point cloud for eval |
| **Evaluation** | — | Chamfer, F@1/2/5 mm, HD95 (world-mm) |
| **N_POINTS** | — | 8192 |

### Key architectural differences from PPCNet v6
- **Output**: 3D occupancy volume (64 cubed) then extract point cloud (vs direct point cloud)
- **Encoder**: Swin Transformer with shifted-window self-attention (vs ResNet-34 CNN)
- **Fusion**: Dimension expanding + cross-attention at bottleneck (vs BiplanarFusion concat+conv)
- **Loss**: DiceCE + KL divergence (vs Chamfer + gap + axial + extent + curvature + SW)
- **No projection matrices**: Swin-X2S does not use calibrated P matrices
- **No iterative refinement**: single forward pass through encoder-decoder


In [ ]:
"""
Swin-X2S Inter-Comparison Baseline
====================================
Liu et al., arXiv 2025 — adapted for lumbar spine point cloud prediction.
Architecture: 2D Swin-T Encoder x2 -> Dimension Expanding -> 3D U-Decoder + CrossAttn -> 64^3
Loss: DiceCE (single-view) + KL divergence (cross-view) — their original
Evaluation: Chamfer, F@1/2/5mm, HD95 in world-mm (our standard metrics)
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import vtk
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# -- GPU MEMORY CAP (MIG-aware) -----------------------------------------------
TARGET_GPU_MEM_GB = 30
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,garbage_collection_threshold:0.6"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Visible device memory (MIG slice or full GPU): {total_gb:.2f} GB")
    frac = min(TARGET_GPU_MEM_GB / total_gb, 0.90)
    if TARGET_GPU_MEM_GB > total_gb:
        print(f"  WARNING: requested {TARGET_GPU_MEM_GB:.1f}GB > visible {total_gb:.2f}GB. "
              f"Capping to {frac*total_gb:.2f}GB (90% of visible device).")
    torch.cuda.set_per_process_memory_fraction(frac, 0)
    print(f"  Memory fraction set: {frac:.3f} (~{frac*total_gb:.2f} GB)")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# -- PATHS --
DATA_ROOT   = Path("./data/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("./inter_comparison_swinx2s")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -- MODEL CONFIG (Swin-X2S settings) --
IMG_SIZE        = 224           # Swin-T default input (ImageNet pretrained)
VOL_SIZE        = 64            # 3D output volume
N_POINTS        = 8192          # same as v6
SWIN_EMBED_DIM  = 96            # Swin-T: C=96
SWIN_DEPTHS     = [2, 2, 6, 2] # Swin-T layer depths
SWIN_HEADS      = [3, 6, 12, 24]
SWIN_WINDOW     = 7

# -- TRAINING CONFIG (Swin-X2S settings) --
BATCH_SIZE      = 2          # reduced: Swin-T 224x224 + 3D decoder OOMs at batch=2 on 9.5GB MIG
NUM_WORKERS     = 4
EPOCHS          = 200           # Swin-X2S: 200 epochs
LR              = 3e-5          # Swin-X2S: AdamW lr=3e-5
WEIGHT_DECAY    = 5e-1          # Swin-X2S: heavy weight decay
WARMUP_EPOCHS   = 20            # Swin-X2S: 20 epochs warmup
GRAD_CLIP       = 1.0
EARLY_STOP_PATIENCE = 60

# -- LOSS WEIGHTS (Swin-X2S original) --
LAMBDA_DICE     = 1.0
LAMBDA_CE       = 1.0
LAMBDA_KL       = 0.1           # KL cross-view loss weight

MAX_EVAL        = 103
CKPT_PATH       = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT       = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG    = LOG_DIR  / "training_log.json"

print("=" * 65)
print("Swin-X2S Inter-Comparison -- Lumbar Spine Point Cloud")
print("=" * 65)
for k, v in dict(
    PAPER="Liu et al., arXiv:2501.05961, 2025",
    INPUT=f"2 x {IMG_SIZE}x{IMG_SIZE} DRR (AP + LP, separate Swin-T encoders)",
    OUTPUT=f"{VOL_SIZE}^3 occupancy volume -> {N_POINTS}-point cloud",
    LOSS=f"DiceCE(dice={LAMBDA_DICE},ce={LAMBDA_CE}) + KL(w={LAMBDA_KL})",
    OPTIMIZER=f"AdamW (lr={LR}, wd={WEIGHT_DECAY}, warmup={WARMUP_EPOCHS}ep)",
    EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
    EVAL_METRICS="Chamfer, F@1/2/5mm, HD95 (world-mm)",
).items():
    print(f"  {k:<18} = {v}")

In [ ]:
# ==============================================================================
# DATA UTILITIES — same coordinate system as PPCNet v6
# ==============================================================================

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points: vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    """Load DRR and resize to Swin-T input resolution (224x224)."""
    img = Image.open(path).convert("L")
    if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
    arr = np.array(img, np.float32) / 255.0
    # ImageNet-style norm for Swin (single channel: use mean of RGB means)
    arr = (arr - 0.485) / 0.229
    return arr

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[None] + c[None]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))

def make_gt_volume(pts_local, vol_size=VOL_SIZE, dilation=1):
    p = np.clip((np.asarray(pts_local, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vol_size).astype(np.int32), 0, vol_size - 1)
    vol = np.zeros((vol_size,) * 3, np.float32)
    for dx in range(-dilation, dilation + 1):
        for dy in range(-dilation, dilation + 1):
            for dz in range(-dilation, dilation + 1):
                vol[np.clip(idx[:, 0] + dx, 0, vol_size - 1),
                    np.clip(idx[:, 1] + dy, 0, vol_size - 1),
                    np.clip(idx[:, 2] + dz, 0, vol_size - 1)] = 1.0
    return vol

def volume_to_points(vol, threshold=0.5, n_points=N_POINTS):
    mask = vol > threshold
    coords = np.argwhere(mask).astype(np.float32)
    if len(coords) == 0:
        return np.random.uniform(-0.5, 0.5, (n_points, 3)).astype(np.float32)
    local = (coords + 0.5) / vol.shape[0] * 2.0 - 1.0
    if len(local) >= n_points:
        idx = np.random.choice(len(local), n_points, replace=False)
        return local[idx]
    else:
        extra = n_points - len(local)
        pad_idx = np.random.choice(len(local), extra, replace=True)
        pad = local[pad_idx] + np.random.randn(extra, 3).astype(np.float32) * 0.01
        return np.concatenate([local, pad], 0)


class LumbarDataset(Dataset):
    """
    Dataset for Swin-X2S inter-comparison.
    Returns AP and LP DRRs separately (each 1x224x224), GT volume, GT points.
    Swin-X2S does NOT use projection matrices.
    """
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids = ids; self.root = Path(root); self.aug = aug

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")

        gt = load_vtk_points(d / "gt_ppc.vtk")
        c  = gt.mean(0).astype(np.float32)
        s  = compute_scale(gt)
        gl = np.clip(pts_to_local(gt, c, s), -1, 1)
        gt_vol = make_gt_volume(gl)

        n = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))

        if self.aug and np.random.rand() < 0.5:
            dap = dap[:, ::-1].copy()
            dlp = dlp[:, ::-1].copy()
            gt_vol = gt_vol[::-1, :, :].copy()

        return {
            "drr_ap":       torch.from_numpy(dap).unsqueeze(0).float(),
            "drr_lp":       torch.from_numpy(dlp).unsqueeze(0).float(),
            "gt_volume":    torch.from_numpy(gt_vol).float(),
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "center":       torch.from_numpy(c).float(),
            "scale":        torch.from_numpy(s).float(),
            "patient_id":   pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")


In [ ]:
# ==============================================================================
# MODEL — Swin-X2S (Liu et al., arXiv 2025)
# ==============================================================================
#
# Encoder: 2D Swin Transformer (Swin-T) x 2 views (AP + LP)
# Bridge: Dimension Expanding Module (2D->3D per pyramid level)
# Decoder: 3D U-shaped decoder with cross-attention at bottleneck
# Output: (B, 64, 64, 64) binary occupancy
#
# Faithful to paper architecture:
#   - Swin-T with ImageNet pretrained weights, adapted for 1-ch greyscale
#   - Hierarchical 4-level feature extraction
#   - Dimension expanding: replicate along orthogonal axis + 1x1x1 3D conv
#   - Cross-attention at bottleneck for biplanar fusion
#   - 3D conv decoder with skip connections
# ==============================================================================

from torchvision.models import swin_t, Swin_T_Weights


class SwinEncoder(nn.Module):
    """
    2D Swin-T encoder adapted for single-channel DRR input.
    Extracts hierarchical features at 4 levels.
    Input: (B, 1, 224, 224)
    Output: list of feature maps at [56x56, 28x28, 14x14, 7x7]
           with channels [96, 192, 384, 768]
    """
    def __init__(self, pretrained=True):
        super().__init__()
        weights = Swin_T_Weights.DEFAULT if pretrained else None
        swin = swin_t(weights=weights)

        # Adapt first patch embedding for 1-channel input
        # Original: Conv2d(3, 96, 4, stride=4) -> average RGB weights
        old_proj = swin.features[0][0]  # PatchEmbed conv
        new_proj = nn.Conv2d(1, old_proj.out_channels,
                             kernel_size=old_proj.kernel_size,
                             stride=old_proj.stride,
                             padding=old_proj.padding, bias=old_proj.bias is not None)
        if pretrained:
            new_proj.weight.data = old_proj.weight.data.mean(dim=1, keepdim=True)
            if old_proj.bias is not None:
                new_proj.bias.data = old_proj.bias.data.clone()
        swin.features[0][0] = new_proj

        # Extract the 4 stages of the Swin backbone
        # features = [PatchEmbed+Norm, Stage1, Stage2, Stage3, Stage4, ...]
        # Swin-T features structure:
        #   [0] = Sequential(Conv2d patch embed, LayerNorm)  -> (B, 96, 56, 56)
        #   [1] = Sequential(SwinTransformerBlock x 2)        -> (B, 96, 56, 56)
        #   [2] = Sequential(PatchMerging)                    -> (B, 192, 28, 28)
        #   [3] = Sequential(SwinTransformerBlock x 2)        -> (B, 192, 28, 28)
        #   [4] = Sequential(PatchMerging)                    -> (B, 384, 14, 14)
        #   [5] = Sequential(SwinTransformerBlock x 6)        -> (B, 384, 14, 14)
        #   [6] = Sequential(PatchMerging)                    -> (B, 768, 7, 7)
        #   [7] = Sequential(SwinTransformerBlock x 2)        -> (B, 768, 7, 7)
        self.features = swin.features
        self.norm = swin.norm  # final LayerNorm
        # Channel counts at each level
        self.channels = [96, 192, 384, 768]

    def forward(self, x):
        feats = []
        out = x
        for i, layer in enumerate(self.features):
            out = layer(out)
            # Collect features after each stage's transformer blocks
            # Stages end at indices 1, 3, 5, 7
            if i in [1, 3, 5, 7]:
                # torchvision Swin-T outputs (B, H, W, C) — channels last.
                # Permute to (B, C, H, W) so AdaptiveAvgPool2d and Conv2d
                # see the correct channel dimension. Without this the model
                # reads H (e.g. 56) as the channel count instead of C (96),
                # causing the "expected 96 channels but got 56" crash.
                feats.append(out.permute(0, 3, 1, 2).contiguous())
        return feats  # [96@56x56, 192@28x28, 384@14x14, 768@7x7]


class DimExpandModule(nn.Module):
    """
    Dimension Expanding Module (Swin-X2S paper Section 3.2).
    Converts 2D feature map to 3D by replicating along the orthogonal axis,
    then refining with a 1x1x1 3D conv.
    """
    def __init__(self, in_ch, out_ch, depth):
        super().__init__()
        self.depth = depth
        self.conv3d = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, feat_2d, axis='depth'):
        """
        feat_2d: (B, C, H, W)
        axis: 'depth' for AP (expand along dim2), 'height' for LP (expand along dim3)
        """
        if axis == 'depth':
            # (B, C, H, W) -> (B, C, D, H, W)
            x = feat_2d.unsqueeze(2).expand(-1, -1, self.depth, -1, -1)
        else:
            # (B, C, H, W) -> (B, C, H, D, W)
            x = feat_2d.unsqueeze(3).expand(-1, -1, -1, self.depth, -1)
        return self.conv3d(x.contiguous())


class CrossAttention3D(nn.Module):
    """
    Cross-attention at bottleneck (Swin-X2S paper Section 3.3).
    AP features attend to LP features and vice versa,
    enabling information exchange between views.
    """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x_query, x_key):
        """x_query, x_key: (B, C, D, H, W)"""
        B, C, D, H, W = x_query.shape
        N = D * H * W

        # Reshape to (B, N, C)
        q = x_query.reshape(B, C, N).permute(0, 2, 1)  # (B, N, C)
        kv = x_key.reshape(B, C, N).permute(0, 2, 1)

        q = self.norm1(q)
        kv = self.norm2(kv)

        Q = self.q_proj(q).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.k_proj(kv).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.v_proj(kv).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ V).permute(0, 2, 1, 3).reshape(B, N, C)
        out = self.out_proj(out)

        # Reshape back to 3D
        out = out.permute(0, 2, 1).reshape(B, C, D, H, W)
        return x_query + out


class ConvBlock3D(nn.Module):
    """Double 3D conv block for decoder."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class SwinX2S(nn.Module):
    """
    Full Swin-X2S model.
    Two Swin-T encoders (AP + LP) -> Dimension Expanding -> Cross-Attention
    -> 3D U-shaped decoder -> 64^3 occupancy.

    Also returns per-view intermediate predictions for the KL cross-view loss.
    """
    def __init__(self):
        super().__init__()
        # Two independent Swin-T encoders
        self.enc_ap = SwinEncoder(pretrained=True)
        self.enc_lp = SwinEncoder(pretrained=True)

        enc_chs = [96, 192, 384, 768]  # Swin-T channels at each level
        # 3D feature volumes spatial sizes for each level
        # Swin output: 56, 28, 14, 7 -> we map to 3D sizes: 32, 16, 8, 4
        vol_sizes = [32, 16, 8, 4]
        dec_chs = [64, 128, 256, 512]  # decoder channel widths

        # Dimension Expanding Modules per level
        self.dim_exp_ap = nn.ModuleList([
            DimExpandModule(enc_chs[i], dec_chs[i], vol_sizes[i]) for i in range(4)])
        self.dim_exp_lp = nn.ModuleList([
            DimExpandModule(enc_chs[i], dec_chs[i], vol_sizes[i]) for i in range(4)])

        # Spatial adapters: resize 2D Swin features to match 3D vol spatial dims
        self.spatial_adapts = nn.ModuleList([
            nn.AdaptiveAvgPool2d(vol_sizes[i]) for i in range(4)])

        # Cross-attention at bottleneck (level 4, deepest)
        self.cross_attn_ap = CrossAttention3D(dec_chs[3], num_heads=8)
        self.cross_attn_lp = CrossAttention3D(dec_chs[3], num_heads=8)

        # Fuse AP + LP at each level: 2*dec_ch -> dec_ch
        self.fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv3d(dec_chs[i] * 2, dec_chs[i], 1, bias=False),
                nn.BatchNorm3d(dec_chs[i]),
                nn.ReLU(inplace=True))
            for i in range(4)])

        # 3D U-shaped decoder: 4^3 -> 8^3 -> 16^3 -> 32^3 -> 64^3
        self.up3 = nn.ConvTranspose3d(dec_chs[3], dec_chs[2], 2, stride=2)
        self.dec3 = ConvBlock3D(dec_chs[2] * 2, dec_chs[2])

        self.up2 = nn.ConvTranspose3d(dec_chs[2], dec_chs[1], 2, stride=2)
        self.dec2 = ConvBlock3D(dec_chs[1] * 2, dec_chs[1])

        self.up1 = nn.ConvTranspose3d(dec_chs[1], dec_chs[0], 2, stride=2)
        self.dec1 = ConvBlock3D(dec_chs[0] * 2, dec_chs[0])

        self.up0 = nn.ConvTranspose3d(dec_chs[0], 32, 2, stride=2)  # 32->64
        self.dec0 = ConvBlock3D(32, 32)

        # Final classification head
        self.head = nn.Conv3d(32, 1, 1)

        # Per-view heads for KL cross-view loss
        self.head_ap = nn.Conv3d(dec_chs[3], 1, 1)
        self.head_lp = nn.Conv3d(dec_chs[3], 1, 1)

    def forward(self, x_ap, x_lp):
        # Encode both views
        feats_ap = self.enc_ap(x_ap)  # [96@56, 192@28, 384@14, 768@7]
        feats_lp = self.enc_lp(x_lp)

        # Dimension expanding at each level
        vols_ap, vols_lp = [], []
        for i in range(4):
            f_ap = self.spatial_adapts[i](feats_ap[i])
            f_lp = self.spatial_adapts[i](feats_lp[i])
            vols_ap.append(self.dim_exp_ap[i](f_ap, axis='depth'))
            vols_lp.append(self.dim_exp_lp[i](f_lp, axis='height'))

        # Cross-attention at bottleneck
        vols_ap[3] = self.cross_attn_ap(vols_ap[3], vols_lp[3])
        vols_lp[3] = self.cross_attn_lp(vols_lp[3], vols_ap[3])

        # Per-view predictions for KL loss (at bottleneck resolution)
        pred_ap_small = torch.sigmoid(self.head_ap(vols_ap[3]))  # (B,1,4,4,4)
        pred_lp_small = torch.sigmoid(self.head_lp(vols_lp[3]))

        # Fuse AP + LP at each level
        fused = []
        for i in range(4):
            f = self.fuse[i](torch.cat([vols_ap[i], vols_lp[i]], dim=1))
            fused.append(f)

        # 3D U-shaped decoder
        x = fused[3]                                           # 512 @ 4^3

        x = self.dec3(torch.cat([self.up3(x), fused[2]], 1))   # 256 @ 8^3
        x = self.dec2(torch.cat([self.up2(x), fused[1]], 1))   # 128 @ 16^3
        x = self.dec1(torch.cat([self.up1(x), fused[0]], 1))   # 64 @ 32^3
        x = self.dec0(self.up0(x))                              # 32 @ 64^3

        pred = torch.sigmoid(self.head(x).squeeze(1))           # (B, 64, 64, 64)

        return pred, pred_ap_small.squeeze(1), pred_lp_small.squeeze(1)


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = SwinX2S(); print(f"Swin-X2S params: {count_params(_m)/1e6:.1f} M"); del _m


In [ ]:
# ==============================================================================
# LOSS FUNCTIONS — Swin-X2S original (Liu et al., 2025)
# ==============================================================================
# 1. Single-view loss: Dice + Cross-Entropy (DiceCE) on main prediction
# 2. Cross-view loss: KL divergence between per-view bottleneck predictions
# Total: L = DiceCE(pred, gt) + lambda_kl * KL(pred_ap, pred_lp)


def dice_loss(pred, gt, smooth=1.0):
    pred_flat = pred.reshape(pred.shape[0], -1)
    gt_flat   = gt.reshape(gt.shape[0], -1)
    intersection = (pred_flat * gt_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + gt_flat.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return (1.0 - dice).mean()


def bce_loss(pred, gt):
    eps = 1e-7
    pred = pred.clamp(eps, 1.0 - eps)
    return F.binary_cross_entropy(pred, gt)


def dice_ce_loss(pred, gt):
    """DiceCE: combination of Dice and cross-entropy (Swin-X2S single-view loss)."""
    return LAMBDA_DICE * dice_loss(pred, gt) + LAMBDA_CE * bce_loss(pred, gt)


def kl_cross_view_loss(pred_ap, pred_lp):
    """
    KL divergence cross-view loss (Swin-X2S paper Eq. 5).
    Forces per-view predictions to agree, improving robustness.
    Symmetric: KL(ap||lp) + KL(lp||ap).
    """
    eps = 1e-7
    p = pred_ap.clamp(eps, 1.0 - eps).reshape(pred_ap.shape[0], -1)
    q = pred_lp.clamp(eps, 1.0 - eps).reshape(pred_lp.shape[0], -1)
    kl_pq = (p * torch.log(p / q)).sum(dim=1).mean()
    kl_qp = (q * torch.log(q / p)).sum(dim=1).mean()
    return 0.5 * (kl_pq + kl_qp)


print("Swin-X2S loss functions defined:")
print(f"  1. DiceCE (single-view)    (dice={LAMBDA_DICE}, ce={LAMBDA_CE})")
print(f"  2. KL cross-view           (w = {LAMBDA_KL})")


In [ ]:
# ==============================================================================
# TRAINING LOOP — Swin-X2S: AdamW, DiceCE + KL, warmup cosine
# ==============================================================================

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}

def chamfer_np(pred, gt):
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th): p, r = (dp<=th).mean(), (dg<=th).mean(); return 2*p*r/(p+r) if p+r>0 else 0.
    return {"chamfer_mm": float(0.5*(dp.mean()+dg.mean())),
            "fscore_1mm": float(fs(1)), "fscore_2mm": float(fs(2)),
            "fscore_5mm": float(fs(5)),
            "hausdorff_95": float(max(np.percentile(dp,95), np.percentile(dg,95)))}


# -- Build model --
model = SwinX2S().to(device)
print(f"Swin-X2S: {count_params(model)/1e6:.1f} M params")

# Enable gradient checkpointing on Swin encoder to reduce backward memory
for layer in model.enc_ap.features:
    if hasattr(layer, 'use_checkpoint'):
        layer.use_checkpoint = True

for layer in model.enc_lp.features:
    if hasattr(layer, 'use_checkpoint'):
        layer.use_checkpoint = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

train_ds     = LumbarDataset(train_ids, aug=True)
val_ds       = LumbarDataset(val_ids,   aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

print(f"Train : {len(train_ds)} samples -> {len(train_loader)} batches/ep")
print(f"Val   : {len(val_ds)} samples -> {len(val_loader)} batches/ep")


def save_ckpt(path, ep, best, hist):
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(),
                "opt": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "epoch": ep, "best_chamfer": best, "history": hist,
                "config": {"method": "Swin-X2S", "paper": "Liu2025"}}, tmp)
    tmp.replace(path)
    print(f"  Saved -> {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["opt"])
    try: scheduler.load_state_dict(st["sched"])
    except: print("  (scheduler state mismatch)")
    ep = st["epoch"] + 1; bc = st.get("best_chamfer", float("inf")); h = st.get("history", [])
    print(f"  Resumed from ep {ep} | best={bc:.3f} mm")
    return ep, bc, h


start_epoch, best_chamfer, history, no_improve = 0, float("inf"), [], 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint -- starting fresh.")

print(f"{'='*65}")
print(f"Swin-X2S training from ep {start_epoch+1}/{EPOCHS}")
print(f"Loss: DiceCE + KL(w={LAMBDA_KL})")
print(f"LR={LR}, warmup {WARMUP_EPOCHS}ep + cosine decay")
print(f"{'='*65}")


for epoch in range(start_epoch, EPOCHS):
    torch.cuda.empty_cache()
    import gc; gc.collect()
    model.train()
    t0 = time.time()
    acc_loss, nb = {}, 0

    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=130)

    for bi, batch in pbar:
        batch = to_dev(batch)
        x_ap   = batch["drr_ap"]
        x_lp   = batch["drr_lp"]
        gt_vol = batch["gt_volume"]

        optimizer.zero_grad(set_to_none=True)

        pred, pred_ap_s, pred_lp_s = model(x_ap, x_lp)

        # Resize GT to bottleneck resolution for KL loss
        gt_small = F.interpolate(
            gt_vol.unsqueeze(1), size=pred_ap_s.shape[1:],
            mode='trilinear', align_corners=False).squeeze(1)

        loss_main = dice_ce_loss(pred, gt_vol)
        loss_ap   = dice_ce_loss(pred_ap_s, gt_small)
        loss_lp   = dice_ce_loss(pred_lp_s, gt_small)
        loss_kl   = LAMBDA_KL * kl_cross_view_loss(pred_ap_s, pred_lp_s)
        loss      = loss_main + 0.5 * (loss_ap + loss_lp) + loss_kl

        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        # -- free graph + fragmented cache after every step --
        del pred, pred_ap_s, pred_lp_s, gt_small, loss_main, loss_ap, loss_lp, loss_kl, loss
        torch.cuda.empty_cache()

        acc_loss["main"]  = acc_loss.get("main",  0) + float(loss_main.detach()) if 'loss_main' in dir() else acc_loss.get("main", 0)
        acc_loss["kl"]    = acc_loss.get("kl",    0) + float(loss_kl.detach())   if 'loss_kl'   in dir() else acc_loss.get("kl",   0)
        acc_loss["total"] = acc_loss.get("total", 0) + float(loss.detach())       if 'loss'      in dir() else acc_loss.get("total",0)
        nb += 1

        if bi % 50 == 0 or bi == len(train_loader):
            g = lambda k: acc_loss.get(k, 0) / max(1, nb)
            pbar.set_postfix_str(
                f"L={g('total'):.3f} main={g('main'):.3f} kl={g('kl'):.4f}")

    scheduler.step()
    elapsed = time.time() - t0

    # -- Validation --
    torch.cuda.empty_cache()
    model.eval(); metrics = []; nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            pred, _, _ = model(batch["drr_ap"], batch["drr_lp"])

            pred_np = pred.cpu().numpy()
            gw_np   = batch["gt_ppc_world"].cpu().numpy()
            centers = batch["center"].cpu().numpy()
            scales  = batch["scale"].cpu().numpy()

            del pred
            torch.cuda.empty_cache()

            for b in range(pred_np.shape[0]):
                if nd >= MAX_EVAL: break
                pred_local = volume_to_points(pred_np[b], threshold=0.5, n_points=N_POINTS)
                pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
                metrics.append(chamfer_np(pred_world, gw_np[b]))
                nd += 1

    if metrics:
        mc = np.mean([m["chamfer_mm"]   for m in metrics])
        f2 = np.mean([m["fscore_2mm"]   for m in metrics])
        hd = np.mean([m["hausdorff_95"] for m in metrics])
        g  = lambda k: acc_loss.get(k, 0) / max(1, nb)
        lr_now = scheduler.get_last_lr()[0]

        print(f"\n[Ep {epoch+1}] {elapsed:.0f}s  lr={lr_now:.2e}  "
              f"Val: Chamfer={mc:.3f}mm  F@2mm={f2:.3f}  HD95={hd:.3f}mm")
        print(f"  Train: main={g('main'):.3f} kl={g('kl'):.4f}")

        rec = {"epoch": epoch+1, "chamfer_mm": mc, "f2": f2, "hd95": hd,
               "train_main": g("main"), "train_kl": g("kl"), "lr": lr_now}
        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)

        if mc < best_chamfer:
            best_chamfer = mc; no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stop: {no_improve} epochs without improvement"); break

print(f"Done. Best val Chamfer: {best_chamfer:.3f} mm")

In [ ]:
# ==============================================================================
# TEST EVALUATION — Swin-X2S: volume -> point cloud -> metrics
# ==============================================================================
import gc, torch

# -- Free ALL GPU memory left over from training ------------------------------
try: del optimizer, scheduler
except NameError: pass
try: del model
except NameError: pass

torch.cuda.synchronize()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info(0)[0] / 1e9
    total = torch.cuda.mem_get_info(0)[1] / 1e9
    print(f"GPU memory free before test load: {free:.2f} / {total:.2f} GB")

# -- Rebuild model fresh, load checkpoint safely via CPU ----------------------
print("Test evaluation...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
    model = SwinX2S().cpu()
    model.load_state_dict(st["model"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val Chamfer {st['best_chamfer']:.3f} mm)")
    del st
    gc.collect()
    model = model.to(device)
else:
    print("  WARNING: No best checkpoint -- using current model state.")
    model = SwinX2S().to(device)

if torch.cuda.is_available():
    free = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"GPU memory free after model load:  {free:.2f} / {total:.2f} GB")

model.eval()

# -- Test DataLoader ----------------------------------------------------------
test_ds     = LumbarDataset(test_ids, aug=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn)

# -- Inference loop -----------------------------------------------------------
all_metrics = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Test', ncols=100):
        batch = to_dev(batch)
        B = batch["drr_ap"].shape[0]
        pred, _, _ = model(batch["drr_ap"], batch["drr_lp"])
        pred_np = pred.cpu().numpy()
        gw_np   = batch["gt_ppc_world"].cpu().numpy()
        centers = batch["center"].cpu().numpy()
        scales  = batch["scale"].cpu().numpy()
        pids    = batch["patient_id"]
        for b in range(B):
            pred_local = volume_to_points(pred_np[b], threshold=0.5, n_points=N_POINTS)
            pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
            m = chamfer_np(pred_world, gw_np[b])
            m["patient_id"] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pred_world, RESULTS_DIR / f"{pids[b]}_pred.vtk")

# -- Print results ------------------------------------------------------------
print(f"\n{'='*65}")
print(f"TEST RESULTS -- Swin-X2S ({len(all_metrics)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm",  "Chamfer mm  "),
                 ("fscore_1mm",  "F-score @1mm"),
                 ("fscore_2mm",  "F-score @2mm"),
                 ("fscore_5mm",  "F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_metrics]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

# -- Save CSV -----------------------------------------------------------------
csv_path = RESULTS_DIR / "test_results_swinx2s.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["patient_id","chamfer_mm",
                                      "fscore_1mm","fscore_2mm",
                                      "fscore_5mm","hausdorff_95"])
    w.writeheader()
    w.writerows(all_metrics)
print(f"\nCSV -> {csv_path}")


In [ ]:
# ==============================================================================
# INTER-COMPARISON TABLE — all 5 methods
# ==============================================================================

import pandas as pd

results = {}

# Swin-X2S
csv = RESULTS_DIR / "test_results_swinx2s.csv"
if csv.exists():
    df = pd.read_csv(csv)
    results["Swin-X2S"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("Swin-X2S results loaded.")

# PPCNet v6
csv = Path("/path/to/inter_comparison_ppcnet_v6/results/test_results_v6_tta.csv")
if csv.exists():
    df = pd.read_csv(csv)
    results["PPCNet v6 (Ours)"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("PPCNet v6 results loaded.")

# 3D-ReVert
csv = Path("/path/to/inter_comparison_3drevert/results/test_results_3drevert.csv")
if csv.exists():
    df = pd.read_csv(csv)
    results["3D-ReVert"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("3D-ReVert results loaded.")

# X2CT-GAN
csv = Path("/path/to/inter_comparison_x2ctgan/results/test_results_x2ctgan.csv")
if csv.exists():
    df = pd.read_csv(csv)
    results["X2CT-GAN"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("X2CT-GAN results loaded.")

# BX2S-Net
csv = Path("/path/to/inter_comparison_bx2snet/results/test_results_bx2snet.csv")
if csv.exists():
    df = pd.read_csv(csv)
    results["BX2S-Net"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("BX2S-Net results loaded.")

if results:
    print(f"\n{'='*90}")
    print("FULL INTER-COMPARISON TABLE (5 methods)")
    print(f"{'='*90}")
    header = f"{'Method':<25} {'Chamfer (mm)':<20} {'F@1mm':<8} {'F@2mm':<8} {'F@5mm':<8} {'HD95 (mm)':<20}"
    print(header)
    print("-" * len(header))
    for method, vals in results.items():
        print(f"{method:<25} {vals['Chamfer']:<20} {vals['F@1mm']:<8} "
              f"{vals['F@2mm']:<8} {vals['F@5mm']:<8} {vals['HD95']:<20}")
    print(f"\nNote: PPCNet v6 (Ours) outputs point clouds directly;")
    print(f"      all baselines output 64^3 volumes -> point cloud extraction.")
else:
    print("No results CSVs found yet -- run training first.")